# 🔬 Universal IoT Data Analysis System
### Auto-detects and analyzes data from ANY simulator

In [ ]:
# Install required packages
!pip install gspread pandas plotly seaborn -q

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import gspread
from google.colab import auth
from google.auth import default
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 🔐 Step 1: Connect to Google Sheets

In [ ]:
# Authenticate
print("🔐 Authenticating with Google...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
# Configuration
# CHANGE THIS TO YOUR SHEET NAME
SHEET_NAME = 'YOUR SHEET NAME_Here'  # Replace with your actual sheet name

In [ ]:
# Load and detect data type
def load_and_detect_data():
    """Load data and auto-detect simulator type"""
    try:
        sheet = gc.open(SHEET_NAME).sheet1
        data = sheet.get_all_records()
        if len(data) == 0:
            print("❌ No data found in sheet!")
            return None, None
        df = pd.DataFrame(data)
        if 'Timestamp' in df.columns:
            df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        columns = df.columns.tolist()
        simulator_type = detect_simulator_type(columns)
        print(f"✅ Detected Simulator: {simulator_type}")
        print(f"📊 Loaded {len(df)} records")
        print(f"📅 Date Range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")
        return df, simulator_type
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None, None


def detect_simulator_type(columns):
    """Detect which simulator based on column names"""
    if 'SoilMoisture' in columns or 'soilMoisture' in columns:
        return 'MUSHROOM_FARM'
    elif 'Pressure' in columns or 'WindSpeed' in columns:
        return 'WEATHER_STATION'
    elif 'LocalTourists' in columns or 'ForeignTourists' in columns:
        return 'TOURISM_MONITOR'
    elif 'TreeCount' in columns or 'FireRisk' in columns:
        return 'FOREST_WATCH'
    else:
        return 'UNKNOWN'


df, simulator_type = load_and_detect_data()

## 📈 Step 2: Universal Data Overview

In [ ]:
# Basic statistics
if df is not None:
    print("\n📊 Dataset Overview")
    print("="*60)
    print(f"Total Records: {len(df)}")
    print(f"Columns: {', '.join(df.columns.tolist())}")
    print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
    print("\n📋 Data Types:")
    print(df.dtypes)
    print("\n📈 Statistical Summary:")
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        print(df[numeric_cols].describe())
    print("\n❓ Missing Values:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("No missing values found! ✨")

## 🎯 Step 3: Simulator-Specific Analysis

In [ ]:
# MUSHROOM FARM ANALYSIS
def analyze_mushroom_farm(df):
    """Specific analysis for Mushroom Farm data"""
    print("\n🍄 MUSHROOM FARM ANALYSIS")
    print("="*60)
    optimal = {'Temperature': (18, 24), 'Humidity': (80, 95), 'CO2': (600, 1000), 'SoilMoisture': (60, 80)}
    print("\n🎯 Time in Optimal Range:")
    for param, (min_val, max_val) in optimal.items():
        if param in df.columns:
            in_range = ((df[param] >= min_val) & (df[param] <= max_val)).sum()
            percentage = (in_range / len(df)) * 100
            print(f"{param}: {percentage:.1f}% of time")
    fig = make_subplots(rows=2, cols=2,
        subplot_titles=('Temperature Trend', 'Humidity Levels', 'CO2 Concentration', 'Soil Moisture'),
        specs=[[{'secondary_y': False}, {'secondary_y': False}], [{'secondary_y': False}, {'secondary_y': False}]])
    fig.add_trace(go.Scatter(x=df['Timestamp'], y=df['Temperature'], name='Temperature', line=dict(color='red')), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['Timestamp'], y=df['Humidity'], name='Humidity', line=dict(color='blue')), row=1, col=2)
    if 'CO2' in df.columns:
        fig.add_trace(go.Scatter(x=df['Timestamp'], y=df['CO2'], name='CO2', line=dict(color='gray')), row=2, col=1)
    if 'SoilMoisture' in df.columns:
        fig.add_trace(go.Scatter(x=df['Timestamp'], y=df['SoilMoisture'], name='Soil Moisture', line=dict(color='green')), row=2, col=2)
    fig.update_layout(height=600, showlegend=False, title_text="🍄 Mushroom Farm Environmental Monitoring")
    fig.show()
    print("\n🌱 Growth Stage Recommendations:")
    avg_temp = df['Temperature'].mean()
    avg_humidity = df['Humidity'].mean()
    if avg_temp >= 24 and avg_humidity < 75:
        print("📍 Conditions suitable for: Spawn Run Stage")
    elif avg_temp >= 18 and avg_temp <= 22 and avg_humidity >= 85:
        print("📍 Conditions suitable for: Pinning Stage")
    else:
        print("📍 Conditions suitable for: Fruiting Stage")

In [ ]:
# WEATHER STATION ANALYSIS
def analyze_weather_station(df):
    """Specific analysis for Weather Station data"""
    print("\n🌤️ WEATHER STATION ANALYSIS")
    print("="*60)
    if 'Province' in df.columns:
        print("\n📍 Data by Province:")
        province_stats = df.groupby('Province').agg({'Temperature': 'mean', 'Humidity': 'mean', 'Rainfall': 'sum'}).round(1)
        print(province_stats)
    fig = make_subplots(rows=3, cols=2,
        subplot_titles=('Temperature', 'Humidity', 'Pressure', 'Wind Speed', 'Rainfall', 'UV Index'))
    params = ['Temperature', 'Humidity', 'Pressure', 'WindSpeed', 'Rainfall', 'UVIndex']
    positions = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2)]
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'yellow']
    for param, pos, color in zip(params, positions, colors):
        if param in df.columns:
            fig.add_trace(go.Scatter(x=df['Timestamp'], y=df[param], name=param, line=dict(color=color)), row=pos[0], col=pos[1])
    fig.update_layout(height=800, showlegend=False, title_text="🌤️ CAR Weather Monitoring Dashboard")
    fig.show()
    print("\n⚠️ Weather Alerts:")
    if 'Temperature' in df.columns:
        if df['Temperature'].max() > 35:
            print("🔥 Heat wave detected!")
        if df['Temperature'].min() < 15:
            print("❄️ Cold weather warning!")
    if 'Rainfall' in df.columns and df['Rainfall'].max() > 20:
        print("🌧️ Heavy rainfall detected!")
    if 'WindSpeed' in df.columns and df['WindSpeed'].max() > 60:
        print("💨 Strong wind warning!")

In [ ]:
# TOURISM MONITOR ANALYSIS
def analyze_tourism_monitor(df):
    """Specific analysis for Tourism Monitor data"""
    print("\n📊 TOURISM MONITOR ANALYSIS")
    print("="*60)
    if 'TotalVisitors' in df.columns:
        print(f"\n👥 Total Visitors Recorded: {df['TotalVisitors'].sum():,}")
        print(f"📈 Average Daily Visitors: {df['TotalVisitors'].mean():.0f}")
        print(f"🏆 Peak Visitors: {df['TotalVisitors'].max():,}")
    if 'Province' in df.columns:
        province_visitors = df.groupby('Province')['TotalVisitors'].sum().sort_values(ascending=False)
        fig = px.pie(values=province_visitors.values, names=province_visitors.index, title='Visitor Distribution by Province')
        fig.show()
    if 'TimePeriod' in df.columns:
        period_stats = df.groupby('TimePeriod').agg({'LocalTourists': 'mean', 'ForeignTourists': 'mean', 'HotelOccupancy': 'mean'}).round(0)
        print("\n📅 Visitor Patterns by Time Period:")
        print(period_stats)
    fig = make_subplots(rows=2, cols=2,
        subplot_titles=('Local vs Foreign Tourists', 'Hotel Occupancy Rate', 'Group Tours', 'Total Visitors Trend'))
    if 'LocalTourists' in df.columns and 'ForeignTourists' in df.columns:
        fig.add_trace(go.Scatter(x=df['Timestamp'], y=df['LocalTourists'], name='Local', line=dict(color='blue')), row=1, col=1)
        fig.add_trace(go.Scatter(x=df['Timestamp'], y=df['ForeignTourists'], name='Foreign', line=dict(color='red')), row=1, col=1)
    if 'HotelOccupancy' in df.columns:
        fig.add_trace(go.Scatter(x=df['Timestamp'], y=df['HotelOccupancy'], name='Occupancy', line=dict(color='green')), row=1, col=2)
    if 'GroupTours' in df.columns:
        fig.add_trace(go.Bar(x=df['Timestamp'], y=df['GroupTours'], name='Groups', marker_color='orange'), row=2, col=1)
    if 'TotalVisitors' in df.columns:
        fig.add_trace(go.Scatter(x=df['Timestamp'], y=df['TotalVisitors'], name='Total', line=dict(color='purple')), row=2, col=2)
    fig.update_layout(height=700, showlegend=True, title_text="📊 CAR Tourism Analytics Dashboard")
    fig.show()
    print("\n💡 Tourism Insights:")
    if 'HotelOccupancy' in df.columns:
        avg_occupancy = df['HotelOccupancy'].mean()
        if avg_occupancy > 80:
            print("🏨 High hotel occupancy - Consider capacity expansion")
        elif avg_occupancy < 50:
            print("🏨 Low hotel occupancy - Marketing campaigns needed")

In [ ]:
# FOREST WATCH ANALYSIS
def analyze_forest_watch(df):
    """Specific analysis for Forest Watch data"""
    print("\n🌲 FOREST WATCH ANALYSIS")
    print("="*60)
    if 'Coverage' in df.columns:
        print(f"\n🌳 Average Forest Coverage: {df['Coverage'].mean():.1f}%")
        print(f"📉 Minimum Coverage: {df['Coverage'].min():.1f}%")
        print(f"📈 Maximum Coverage: {df['Coverage'].max():.1f}%")
    if 'FireRisk' in df.columns:
        high_risk_days = (df['FireRisk'] > 70).sum()
        print(f"\n🔥 High Fire Risk Days: {high_risk_days} ({high_risk_days/len(df)*100:.1f}%)")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df['Timestamp'], y=df['FireRisk'], mode='lines', name='Fire Risk',
                                 line=dict(color='red', width=2), fill='tozeroy'))
        fig.add_hrect(y0=70, y1=100, fillcolor="red", opacity=0.2, annotation_text="Danger Zone")
        fig.update_layout(title='🔥 Fire Risk Timeline', xaxis_title='Time', yaxis_title='Fire Risk Level', height=400)
        fig.show()
    fig = make_subplots(rows=3, cols=2,
        subplot_titles=('Temperature', 'Moisture', 'Air Quality', 'Tree Count', 'Growth Rate', 'Wildlife Species'))
    params = [('Temperature', 'red', (1,1)), ('Moisture', 'blue', (1,2)), ('AirQuality', 'gray', (2,1)),
              ('TreeCount', 'green', (2,2)), ('GrowthRate', 'orange', (3,1)), ('Species', 'purple', (3,2))]
    for param, color, pos in params:
        if param in df.columns:
            fig.add_trace(go.Scatter(x=df['Timestamp'], y=df[param], name=param, line=dict(color=color)), row=pos[0], col=pos[1])
    fig.update_layout(height=800, showlegend=False, title_text="🌲 Forest Environmental Monitoring")
    fig.show()
    print("\n🚨 Conservation Alerts:")
    if 'TreeCount' in df.columns:
        tree_loss = df['TreeCount'].iloc[0] - df['TreeCount'].iloc[-1]
        if tree_loss > 0:
            print(f"⚠️ Tree loss detected: {tree_loss} trees")
    if 'AirQuality' in df.columns and df['AirQuality'].mean() > 150:
        print("💨 Poor air quality detected - Investigation needed")

In [ ]:
# Run appropriate analysis based on detected type
if df is not None and simulator_type != 'UNKNOWN':
    if simulator_type == 'MUSHROOM_FARM':
        analyze_mushroom_farm(df)
    elif simulator_type == 'WEATHER_STATION':
        analyze_weather_station(df)
    elif simulator_type == 'TOURISM_MONITOR':
        analyze_tourism_monitor(df)
    elif simulator_type == 'FOREST_WATCH':
        analyze_forest_watch(df)
else:
    print("⚠️ Could not determine simulator type or no data available")

## 📊 Step 4: Universal Visualizations (Works for All Types)

In [ ]:
# Time series analysis for any numeric data
if df is not None:
    print("\n⏰ TIME SERIES ANALYSIS")
    print("="*60)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        fig = go.Figure()
        for col in numeric_cols[:5]:
            fig.add_trace(go.Scatter(x=df['Timestamp'], y=df[col], mode='lines', name=col))
        fig.update_layout(title='Time Series Data Overview', xaxis_title='Time', yaxis_title='Values',
                          hovermode='x unified', height=500)
        fig.update_xaxes(rangeslider_visible=True)
        fig.show()

In [ ]:
# Correlation analysis
if df is not None:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) > 1:
        print("\n🔗 CORRELATION ANALYSIS")
        print("="*60)
        corr_matrix = df[numeric_cols].corr()
        fig = px.imshow(corr_matrix, labels=dict(color="Correlation"), x=numeric_cols, y=numeric_cols,
                        color_continuous_scale='RdBu', aspect="auto", title="Parameter Correlation Heatmap")
        fig.update_layout(height=600)
        fig.show()
        print("\n💪 Strong Correlations (>0.7):")
        for i in range(len(numeric_cols)):
            for j in range(i+1, len(numeric_cols)):
                if abs(corr_matrix.iloc[i, j]) > 0.7:
                    print(f"{numeric_cols[i]} <-> {numeric_cols[j]}: {corr_matrix.iloc[i, j]:.2f}")

In [ ]:
# Hourly patterns analysis
if df is not None and 'Timestamp' in df.columns:
    print("\n🕐 HOURLY PATTERNS")
    print("="*60)
    df['Hour'] = df['Timestamp'].dt.hour
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'Hour' in numeric_cols:
        numeric_cols.remove('Hour')
    hourly_avg = df.groupby('Hour')[numeric_cols[:4]].mean()
    fig = go.Figure()
    for col in hourly_avg.columns:
        fig.add_trace(go.Scatter(x=hourly_avg.index, y=hourly_avg[col], mode='lines+markers', name=col))
    fig.update_layout(title='Average Hourly Patterns', xaxis_title='Hour of Day', yaxis_title='Average Values', height=400)
    fig.show()

## 🎯 Step 5: Anomaly Detection (Universal)

In [ ]:
# Detect anomalies in any numeric data
def detect_anomalies(df):
    """Universal anomaly detection using statistical methods"""
    print("\n🔍 ANOMALY DETECTION")
    print("="*60)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    anomalies_found = {}
    for col in numeric_cols:
        window = min(20, len(df) // 10) or 1
        rolling_mean = df[col].rolling(window=window, min_periods=1).mean()
        rolling_std = df[col].rolling(window=window, min_periods=1).std()
        upper_bound = rolling_mean + (3 * rolling_std)
        lower_bound = rolling_mean - (3 * rolling_std)
        anomalies = ((df[col] > upper_bound) | (df[col] < lower_bound))
        anomaly_count = anomalies.sum()
        if anomaly_count > 0:
            anomalies_found[col] = anomaly_count
            print(f"⚠️ {col}: {anomaly_count} anomalies ({anomaly_count/len(df)*100:.1f}%)")
    if not anomalies_found:
        print("✅ No significant anomalies detected!")
    return anomalies_found


if df is not None:
    anomalies = detect_anomalies(df)

## 📈 Step 6: Predictive Insights

In [ ]:
# Generate insights based on data patterns
def generate_insights(df, simulator_type):
    """Generate actionable insights based on data patterns"""
    print("\n💡 ACTIONABLE INSIGHTS")
    print("="*60)
    if simulator_type == 'MUSHROOM_FARM':
        if 'Temperature' in df.columns and 'Humidity' in df.columns:
            optimal_temp = (df['Temperature'] >= 18) & (df['Temperature'] <= 24)
            optimal_humidity = (df['Humidity'] >= 80) & (df['Humidity'] <= 95)
            optimal_conditions = (optimal_temp & optimal_humidity).sum() / len(df) * 100
            print(f"🍄 Optimal growing conditions maintained {optimal_conditions:.1f}% of time")
            if optimal_conditions < 70:
                print("   -> Consider automated climate control system")
    elif simulator_type == 'WEATHER_STATION':
        if 'Rainfall' in df.columns:
            total_rainfall = df['Rainfall'].sum()
            print(f"☔ Total rainfall: {total_rainfall:.1f}mm")
            if total_rainfall < 50:
                print("   -> Drought conditions - water conservation recommended")
            elif total_rainfall > 200:
                print("   -> Excess rainfall - flood prevention measures needed")
    elif simulator_type == 'TOURISM_MONITOR':
        if 'TotalVisitors' in df.columns:
            visitor_trend = np.polyfit(range(len(df)), df['TotalVisitors'], 1)[0]
            if visitor_trend > 0:
                print(f"📈 Tourism is growing at {visitor_trend:.1f} visitors/day")
                print("   -> Consider expanding tourist facilities")
            else:
                print(f"📉 Tourism is declining at {abs(visitor_trend):.1f} visitors/day")
                print("   -> Marketing campaigns recommended")
    elif simulator_type == 'FOREST_WATCH':
        if 'FireRisk' in df.columns:
            high_risk_periods = (df['FireRisk'] > 70).sum()
            if high_risk_periods > len(df) * 0.1:
                print("🔥 High fire risk detected frequently")
                print("   -> Increase fire prevention patrols")
                print("   -> Install additional fire detection sensors")


if df is not None:
    generate_insights(df, simulator_type)

## 📊 Step 7: Export Processed Data

In [ ]:
# Export to CSV with additional features
def export_enhanced_data(df, simulator_type):
    """Export data with additional calculated features"""
    df_export = df.copy()
    if 'Timestamp' in df_export.columns:
        df_export['Hour'] = df_export['Timestamp'].dt.hour
        df_export['DayOfWeek'] = df_export['Timestamp'].dt.dayofweek
        df_export['Month'] = df_export['Timestamp'].dt.month
    if simulator_type == 'MUSHROOM_FARM':
        if 'Temperature' in df_export.columns and 'Humidity' in df_export.columns:
            df_export['VPD'] = 0.611 * np.exp(17.27 * df_export['Temperature'] /
                              (df_export['Temperature'] + 237.3)) * (1 - df_export['Humidity']/100)
    elif simulator_type == 'WEATHER_STATION':
        if 'Temperature' in df_export.columns and 'Humidity' in df_export.columns:
            df_export['HeatIndex'] = df_export['Temperature'] + 0.5 * (df_export['Humidity'] - 60)
    elif simulator_type == 'TOURISM_MONITOR':
        if 'LocalTourists' in df_export.columns and 'ForeignTourists' in df_export.columns:
            df_export['ForeignRatio'] = df_export['ForeignTourists'] / (df_export['LocalTourists'] + df_export['ForeignTourists'] + 1)
    elif simulator_type == 'FOREST_WATCH':
        if 'Temperature' in df_export.columns and 'Moisture' in df_export.columns:
            df_export['FireDangerIndex'] = (100 - df_export['Moisture']) * (df_export['Temperature'] / 40)
    filename = f'{simulator_type.lower()}_analysis_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    df_export.to_csv(filename, index=False)
    print(f"\n✅ Data exported to: {filename}")
    print(f"   Rows: {len(df_export)}")
    print(f"   Columns: {len(df_export.columns)}")
    return df_export


if df is not None:
    df_enhanced = export_enhanced_data(df, simulator_type)

## 📋 Summary Report

In [ ]:
# Generate final summary
if df is not None:
    print("\n" + "="*60)
    print("📋 ANALYSIS SUMMARY REPORT")
    print("="*60)
    print(f"Simulator Type: {simulator_type}")
    print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Data Points Analyzed: {len(df)}")
    print(f"Time Period: {df['Timestamp'].min()} to {df['Timestamp'].max()}")
    print(f"Duration: {(df['Timestamp'].max() - df['Timestamp'].min()).days} days")
    print("\n🎯 Key Metrics:")
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols[:5]:
        print(f"   {col}:")
        print(f"      Mean: {df[col].mean():.2f}")
        print(f"      Min: {df[col].min():.2f}")
        print(f"      Max: {df[col].max():.2f}")
    print("\n✅ Analysis Complete!")
else:
    print("❌ No data available for analysis")
    print("Please ensure your simulator is sending data to Google Sheets")